# Orchestration

**Question:** Can a per-class-routed, confidence-aware framework beat any single model
while staying tunable by Kadaster?

**Deliverables (this notebook):**
1. Per-deed score / feature table (per-method top-1 vs top-2 score, descriptive only)
2. Fixed-rule routing: each code → its best method; ablation vs each method alone
3. Confidence escalation using a multi-label-aware metric (weakest accepted vs strongest rejected): precision–coverage curve
4. Disagreement signal: how well BERT vs regex disagreement predicts errors
5. `policy.yaml` + `explain()` view
6. `orchestration.parquet` written via the shared contract

## 0. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "..")   # make shared/ importable from notebooks/

import numpy as np
import pandas as pd
import yaml

from sklearn.metrics import f1_score, precision_score

from shared.config  import TEST_PATH, POLICY_PATH, RESULTS, PREDS_DIR
from shared.data    import load_jsonl
from shared.labels  import get_mlb
from shared.io      import read_predictions, write_predictions, predictions_to_matrix
from shared.harness import evaluate, per_class_matrix, print_summary

print("Imports OK")

## 1. Label Space & Ground Truth

In [ ]:
mlb, classes, num_classes = get_mlb()

# Test ground truth (touched only for final evaluation)
test_df  = load_jsonl(TEST_PATH)
y_true   = mlb.transform(test_df["rechtsfeitcodes"]).astype(np.int32)
test_ids = test_df["akteId"].tolist()
print(f"Test set   : {len(test_ids)} deeds  |  {num_classes} codes")
print(f"y_true     : shape={y_true.shape}  positive={int(y_true.sum())} ({y_true.sum()/y_true.size:.1%})")

# Val ground truth (used to fit routing map + thresholds — fixes test-set leakage)
from shared.config import TRAIN_PATH
from shared.splits import load_split
train_full_df = load_jsonl(TRAIN_PATH)
_, val_idx    = load_split()
val_df  = train_full_df.iloc[val_idx].reset_index(drop=True)
y_val   = mlb.transform(val_df["rechtsfeitcodes"]).astype(np.int32)
val_ids = val_df["akteId"].tolist()
print(f"Val set    : {len(val_ids)} deeds  |  positive={int(y_val.sum())}")

## 2. Load Predictions

`METHOD1` defaults to `METHOD1` (current best model).
Switch to `METHOD1` once the calibrated `bert.parquet` is available.

In [ ]:
METHOD1 = "eurobert"
METHOD2 = "regex_hybrid"

METHOD1_df  = read_predictions(METHOD1)
METHOD2_df = read_predictions(METHOD2)


def align_to_ids(df: pd.DataFrame, target_ids: list, classes: list):
    """Long-format DataFrame -> (N, C) matrices aligned to target_ids ordering."""
    ids_out, scores, pred = predictions_to_matrix(df, classes)
    id2row = {a: i for i, a in enumerate(ids_out)}
    idx = [id2row[a] for a in target_ids]
    return scores[idx].astype(np.float32), pred[idx].astype(bool)


# Test predictions
METHOD1_scores,  METHOD1_pred  = align_to_ids(METHOD1_df,  test_ids, classes)
METHOD2_scores, METHOD2_pred = align_to_ids(METHOD2_df, test_ids, classes)

# Val predictions (for fitting routing + thresholds)
METHOD1_val_df  = read_predictions("eurobert_val")
METHOD2_val_df = read_predictions("regex_hybrid_val")

METHOD1_val_scores,  METHOD1_val_pred  = align_to_ids(METHOD1_val_df,  val_ids, classes)
METHOD2_val_scores, METHOD2_val_pred = align_to_ids(METHOD2_val_df, val_ids, classes)

print(f"BERT  test predicted : {METHOD1_pred.sum()}  |  val predicted : {METHOD1_val_pred.sum()}")
print(f"Regex test predicted : {METHOD2_pred.sum()}  |  val predicted : {METHOD2_val_pred.sum()}")

## 3. Per-Deed Feature Table

For each deed and method: top-1 score, top-2 score, and the gap between them.

**Note:** this top-1/top-2 gap is purely a descriptive diagnostic of each method's raw
score distribution. It is *not* used as the escalation signal. In a multi-label setting
a small gap between the top two scores can simply mean "two correct positive labels with
similar confidence" — not uncertainty. Section 5 below defines a metric that respects
that distinction.

In [ ]:
def deed_features(scores: np.ndarray, pred: np.ndarray, ids: list, prefix: str) -> pd.DataFrame:
    C       = scores.shape[1]
    top_idx = np.argsort(scores, axis=1)[:, ::-1]
    top1_s  = scores[np.arange(len(ids)), top_idx[:, 0]]
    top2_s  = scores[np.arange(len(ids)), top_idx[:, 1]] if C > 1 else np.zeros(len(ids))
    top1_c  = [classes[i] for i in top_idx[:, 0]]
    return pd.DataFrame({
        "akteId":               ids,
        f"{prefix}_top1_code":  top1_c,
        f"{prefix}_top1_score": top1_s.round(4),
        f"{prefix}_top2_score": top2_s.round(4),
        f"{prefix}_margin":     (top1_s - top2_s).round(4),
        f"{prefix}_n_pred":     pred.sum(axis=1).astype(int),
    })


METHOD1_feats  = deed_features(METHOD1_scores,  METHOD1_pred,  test_ids, METHOD1)
METHOD2_feats = deed_features(METHOD2_scores, METHOD2_pred, test_ids, METHOD2)
deed_table  = METHOD1_feats.merge(METHOD2_feats, on="akteId")

score_cols = [c for c in deed_table.columns if c != "akteId" and "code" not in c]
print(deed_table[score_cols].describe().T[["mean", "std", "min", "max"]].round(3))
deed_table.head(5)

## 4. Per-Class Matrix & Fixed-Rule Routing

### 4a. Evaluate methods individually

In [ ]:
m_METHOD1  = evaluate(y_true, METHOD1_pred.astype(np.int32),  classes, method=METHOD1)
m_METHOD2 = evaluate(y_true, METHOD2_pred.astype(np.int32), classes, method=METHOD2)
print_summary([m_METHOD1, m_METHOD2])

### 4b. Per-class F1 matrix

For each code: which method wins?

In [ ]:
matrix = per_class_matrix(
    {
        METHOD1: (y_true, METHOD1_pred.astype(np.int32)),
        METHOD2:     (y_true, METHOD2_pred.astype(np.int32)),
    },
    classes,
)

METHOD1_f1_col  = f"{METHOD1}_f1"
METHOD2_f1_col = f"{METHOD2}_f1"

METHOD1_f1s  = matrix.get(METHOD1_f1_col,  pd.Series([0.0] * len(matrix))).fillna(0.0)
METHOD2_f1s = matrix.get(METHOD2_f1_col, pd.Series([0.0] * len(matrix))).fillna(0.0)
print(f"Codes where Method 1 wins  : {int((METHOD1_f1s >= METHOD2_f1s).sum())}")
print(f"Codes where Method2 wins : {int((METHOD2_f1s > METHOD1_f1s).sum())}")
matrix.head(15)

### 4c. Build routed predictions

For each code, route to the method with the higher test-set F1.
Build `y_pred_routed` by selecting the designated method's prediction column per code.

In [ ]:
# Fit routing_map on VAL per-class F1 (not test — fixes leakage issue #1)
val_matrix = per_class_matrix(
    {
        METHOD1: (y_val, METHOD1_val_pred.astype(np.int32)),
        METHOD2:     (y_val, METHOD2_val_pred.astype(np.int32)),
    },
    classes,
    save=False,   # don't overwrite the test-based per_class_matrix.csv
)

METHOD1_f1_col  = f"{METHOD1}_f1"
METHOD2_f1_col = f"{METHOD2}_f1"

routing_map = {}
for _, row in val_matrix.iterrows():
    code = row["code"]
    b_f1 = float(row.get(METHOD1_f1_col,  0.0) or 0.0)
    r_f1 = float(row.get(METHOD2_f1_col, 0.0) or 0.0)
    routing_map[code] = METHOD1 if b_f1 >= r_f1 else METHOD2

METHOD1_routes  = sum(1 for v in routing_map.values() if v == METHOD1)
METHOD2_routes = len(routing_map) - METHOD1_routes
print(f"Routing (fit on val): {METHOD1_routes} codes -> METHOD1,  {METHOD2_routes} codes -> METHOD2")

# Apply to test
routing_idx = np.array([0 if routing_map.get(c) == METHOD1 else 1 for c in classes])

y_pred_routed = np.where(
    routing_idx[np.newaxis, :] == 0, METHOD1_pred, METHOD2_pred
).astype(np.int32)

m_routed = evaluate(y_true, y_pred_routed, classes, method="routed")
print()
print_summary([m_METHOD1, m_METHOD2, m_routed])

## 5. Confidence Escalation (multi-label-aware)

A global "top1 vs top2 score" margin implicitly assumes there's one right answer per
deed — like a single-label softmax. But this is a multi-label problem: a deed can have
several true codes, and two real positives sitting close in score is not uncertainty,
it's two confident labels. The old metric would have flagged that case as "uncertain"
incorrectly.

Instead, for each deed we look at its own predicted set:
- **`min_pred_score`** — the weakest of the codes we are about to assign (the "weakest link" of what we're committing to)
- **`max_rej_score`** — the strongest code we decided *not* to assign
- **`gap`** = `min_pred_score - max_rej_score` — how cleanly separated the accepted codes are from the rejected ones

A deed with **zero predicted codes** is always uncertain — there's nothing to be confident about, so it's escalated regardless of score.

**Uncertain definition (per deed):**
- No codes were predicted at all, OR
- The weakest accepted code's score is below `AUTO_THRESHOLD`, OR
- The accept/reject gap is below `MIN_GAP`

We sweep `AUTO_THRESHOLD` over `min_pred_score` and plot **precision vs coverage** to find
the operating point: the highest fraction of deeds auto-handled at ≥ 90% micro-precision.

In [ ]:
# Per-deed routed scores: pick score from each code's assigned method
routed_scores = np.where(
    routing_idx[np.newaxis, :] == 0, METHOD1_scores, METHOD2_scores
).astype(np.float32)

# Multi-label-aware confidence signal (see markdown above for why this
# replaces a global top1-vs-top2 margin).
pred_mask = y_pred_routed.astype(bool)

min_pred_score = np.where(pred_mask, routed_scores, np.inf).min(axis=1)
max_rej_score  = np.where(~pred_mask, routed_scores, -np.inf).max(axis=1)

no_pred = ~pred_mask.any(axis=1)
min_pred_score = np.where(no_pred, np.nan, min_pred_score)

gap = min_pred_score - max_rej_score

deed_conf_df = pd.DataFrame({
    "akteId":         test_ids,
    "min_pred_score": min_pred_score,
    "max_rej_score":  max_rej_score,
    "gap":            gap,
    "no_pred":        no_pred,
})

print(f"Deeds with zero predicted codes: {int(no_pred.sum())}")
print(deed_conf_df[["min_pred_score", "max_rej_score", "gap"]].describe().round(4))

In [ ]:
# Compute val routing scores + confidence signals for threshold sweep
val_routed_scores = np.where(
    routing_idx[np.newaxis, :] == 0, METHOD1_val_scores, METHOD2_val_scores
).astype(np.float32)
val_routed_pred = np.where(
    routing_idx[np.newaxis, :] == 0, METHOD1_val_pred, METHOD2_val_pred
).astype(np.int32)

val_pred_mask       = val_routed_pred.astype(bool)
val_no_pred         = ~val_pred_mask.any(axis=1)
val_has_pred        = ~val_no_pred
val_min_pred_score  = np.where(val_pred_mask, val_routed_scores, np.inf).min(axis=1)
val_min_pred_score  = np.where(val_no_pred, np.nan, val_min_pred_score)
val_has_pred        = ~val_no_pred
val_min_pred_vals   = np.nan_to_num(val_min_pred_score, nan=-1.0)

# Sweep AUTO_THRESHOLD on VAL (not test — fixes leakage issue #1)
thresholds = np.linspace(0.05, 0.95, 80)
sweep_rows = []

for t in thresholds:
    auto_mask = val_has_pred & (val_min_pred_vals >= t)
    n_auto = int(auto_mask.sum())
    if n_auto == 0:
        sweep_rows.append({"threshold": t, "coverage": 0.0,
                           "micro_prec": float("nan"), "macro_f1": float("nan")})
        continue
    idx = np.where(auto_mask)[0]
    mp  = precision_score(y_val[idx], val_routed_pred[idx], average="micro", zero_division=0)
    mf1 = f1_score(y_val[idx],        val_routed_pred[idx], average="macro",  zero_division=0)
    sweep_rows.append({"threshold": t, "coverage": n_auto / len(val_ids),
                       "micro_prec": mp, "macro_f1": mf1})

sweep_df = pd.DataFrame(sweep_rows)

at_90 = sweep_df.dropna().query("micro_prec >= 0.90")
if not at_90.empty:
    op = at_90.sort_values("coverage", ascending=False).iloc[0]
    AUTO_THRESHOLD = float(op["threshold"])
    print(
        f"Operating point (val micro-prec >= 0.90): "
        f"threshold={AUTO_THRESHOLD:.3f}  coverage={op['coverage']:.1%}  "
        f"val micro-prec={op['micro_prec']:.4f}  val macro-F1={op['macro_f1']:.4f}"
    )
else:
    AUTO_THRESHOLD = 0.50
    print(f"No val point reaches 0.90 precision; defaulting AUTO_THRESHOLD={AUTO_THRESHOLD}")


In [ ]:
# Precision-Coverage curve sweeping MIN_GAP on VAL (full escalation policy)
# AUTO_THRESHOLD is already fixed from the score sweep above.

# Compute val gap signal (min_pred_score - max_rej_score per deed)
val_max_rej_score = np.where(~val_pred_mask, val_routed_scores, -np.inf).max(axis=1)
val_gap = np.where(
    val_has_pred,
    np.nan_to_num(val_min_pred_score, nan=0.0) - val_max_rej_score,
    np.nan,
)
val_gap_safe = np.nan_to_num(val_gap, nan=-1.0)

gap_thresholds = np.linspace(0.0, 0.5, 100)
gap_sweep_rows = []

for g in gap_thresholds:
    val_uncertain = (
        val_no_pred |
        (val_min_pred_vals < AUTO_THRESHOLD) |
        (val_gap_safe < g)
    )
    auto_idx = np.where(~val_uncertain)[0]
    n_auto = len(auto_idx)
    coverage = n_auto / len(val_ids)
    if n_auto == 0:
        gap_sweep_rows.append({"min_gap": g, "coverage": coverage,
                               "micro_prec": float("nan"), "macro_f1": float("nan")})
        continue
    mp  = precision_score(y_val[auto_idx], val_routed_pred[auto_idx],
                          average="micro", zero_division=0)
    mf1 = f1_score(y_val[auto_idx], val_routed_pred[auto_idx],
                   average="macro", zero_division=0)
    gap_sweep_rows.append({"min_gap": g, "coverage": coverage,
                           "micro_prec": mp, "macro_f1": mf1})

gap_sweep_df = pd.DataFrame(gap_sweep_rows)

op_row = gap_sweep_df.iloc[(gap_sweep_df["min_gap"] - 0.1).abs().argmin()]
print(f"At MIN_GAP=0.1: coverage={op_row['coverage']:.1%}  "
      f"val micro-prec={op_row['micro_prec']:.4f}  val macro-F1={op_row['macro_f1']:.4f}")


## 6. Disagreement Signal

Flag deeds where BERT and regex predict different label sets, then check whether
disagreement predicts errors — an orthogonal escalation signal independent of confidence scores.

In [ ]:
agree_mask    = (METHOD1_pred == METHOD2_pred).all(axis=1)
disagree_mask = ~agree_mask
print(f"Disagreement rate : {disagree_mask.mean():.1%}  ({disagree_mask.sum()} / {len(test_ids)} deeds)")

# Does disagreement predict errors on the routed prediction?
fully_correct = (y_pred_routed == y_true).all(axis=1)
for name, mask in [("Agree", agree_mask), ("Disagree", disagree_mask)]:
    n   = int(mask.sum())
    pct = fully_correct[mask].mean() if n > 0 else 0.0
    print(f"  {name:10s} ({n:5d} deeds):  fully-correct={pct:.1%}  has-error={1-pct:.1%}")

In [ ]:
# Per-code disagreement rate
code_disagree = pd.DataFrame([
    {"code": classes[j],
     "disagree_rate": float((METHOD1_pred[:, j] != METHOD2_pred[:, j]).mean())}
    for j in range(len(classes))
]).sort_values("disagree_rate", ascending=False).reset_index(drop=True)

print("Top-10 codes by METHOD1/METHOD2 disagreement rate:")
print(code_disagree.head(10).to_string(index=False))


## 7. Adjustable Policy Table

`policy.yaml` is the single file Kadaster can edit to change routing without touching code:
- **`global.min_score`** — deeds whose weakest accepted code is below this are escalated
- **`global.min_gap`** — deeds whose accept/reject gap is below this are escalated
- **`per_code[code].method`** — which method handles this code
- **`per_code[code].escalate_if_uncertain`** — toggle per-code escalation

Deeds with zero predicted codes are always escalated, regardless of these settings.

In [ ]:
MIN_GAP = 0.1

no_pred_vals = deed_conf_df["no_pred"].values
min_pred_safe = np.nan_to_num(deed_conf_df["min_pred_score"].values, nan=-1.0)
gap_safe      = np.nan_to_num(deed_conf_df["gap"].values,            nan=-1.0)

uncertain_mask = (
    no_pred_vals |
    (min_pred_safe < AUTO_THRESHOLD) |
    (gap_safe      < MIN_GAP)
)
print(f"Uncertain -> escalated : {uncertain_mask.sum()} deeds ({uncertain_mask.mean():.1%})")
print(f"  of which zero-predicted : {int(no_pred_vals.sum())} deeds")
print(f"Auto-handled           : {(~uncertain_mask).sum()} deeds ({(~uncertain_mask).mean():.1%})")

In [ ]:
# Export disagreement instances to Excel for manual review
disagree_idx = np.where(disagree_mask)[0]

rows = []
for i in disagree_idx:
    true_codes      = [classes[j] for j in range(len(classes)) if y_true[i, j]]
    bert_codes      = [classes[j] for j in range(len(classes)) if METHOD1_pred[i, j]]
    regex_codes     = [classes[j] for j in range(len(classes)) if METHOD2_pred[i, j]]
    routed_codes    = [classes[j] for j in range(len(classes)) if y_pred_routed[i, j]]
    disagreeing_on  = [classes[j] for j in range(len(classes)) if METHOD1_pred[i, j] != METHOD2_pred[i, j]]
    rows.append({
        "akteId":         test_ids[i],
        "true_codes":     ", ".join(str(c) for c in true_codes),
        "eurobert_codes": ", ".join(str(c) for c in bert_codes),
        "regex_codes":    ", ".join(str(c) for c in regex_codes),
        "routed_codes":   ", ".join(str(c) for c in routed_codes),
        "disagreeing_on": ", ".join(str(c) for c in disagreeing_on),
        "escalated":      bool(uncertain_mask[i]),
        "routed_correct": bool((y_pred_routed[i] == y_true[i]).all()),
    })

disagree_df = pd.DataFrame(rows)
out_path = RESULTS / "disagreement_instances.xlsx"
disagree_df.to_excel(out_path, index=False)
print(f"Saved {len(disagree_df)} disagreement instances -> {out_path}")
disagree_df.head()


In [ ]:
print(f"AUTO_THRESHOLD : {AUTO_THRESHOLD:.3f}")
print(f"MIN_GAP        : {MIN_GAP}")
print()

# Break down which condition drives each escalation
cond_no_pred   = no_pred_vals
cond_low_score = (~no_pred_vals) & (min_pred_safe < AUTO_THRESHOLD)
cond_low_gap   = (~no_pred_vals) & (min_pred_safe >= AUTO_THRESHOLD) & (gap_safe < MIN_GAP)
cond_multi     = (~no_pred_vals) & (min_pred_safe < AUTO_THRESHOLD) & (gap_safe < MIN_GAP)

N = len(test_ids)
print(f"Escalated by no_pred only          : {cond_no_pred.sum():4d}  ({cond_no_pred.mean():.1%})")
print(f"Escalated by low score only        : {cond_low_score.sum():4d}  ({cond_low_score.mean():.1%})")
print(f"Escalated by low gap only          : {cond_low_gap.sum():4d}  ({cond_low_gap.mean():.1%})")
print(f"Escalated by both score + gap      : {cond_multi.sum():4d}  ({cond_multi.mean():.1%})")
print(f"Total escalated                    : {uncertain_mask.sum():4d}  ({uncertain_mask.mean():.1%})")

In [ ]:
policy = {
    "version": "1.0",
    "description": (
        "Per-code routing policy for Kadaster deed classification. "
        "Edit method/threshold per code to tune. "
        "A deed is escalated when it has zero predicted codes, OR the weakest "
        "accepted code's score is below global.min_score, OR the gap between "
        "the weakest accepted code and the strongest rejected code is below "
        "global.min_gap."
    ),
    "global": {
        "min_score": round(AUTO_THRESHOLD, 3),
        "min_gap":   MIN_GAP,
    },
    "per_code": {},
}

for j, code in enumerate(classes):
    method  = routing_map.get(code, METHOD1)
    row_hit = val_matrix[val_matrix["code"] == code]
    f1_val  = 0.0
    if not row_hit.empty:
        f1_col = f"{method}_f1"
        f1_val = float(row_hit.iloc[0].get(f1_col, 0.0) or 0.0)
    dr  = float((METHOD1_pred[:, j] != METHOD2_pred[:, j]).mean())
    key = int(code) if str(code).isdigit() else code
    policy["per_code"][key] = {
        "method":                method,
        "f1":                    round(f1_val, 4),
        "disagree_rate":         round(dr, 4),
        "escalate_if_uncertain": True,
    }

POLICY_PATH.parent.mkdir(parents=True, exist_ok=True)
with POLICY_PATH.open("w", encoding="utf-8") as fh:
    yaml.dump(policy, fh, sort_keys=False, allow_unicode=True)
print(f"Policy saved -> {POLICY_PATH}")

### 7a. Explain view

`explain(deed_id)` shows per-code: which method handled it, both scores,
whether it was predicted, and whether it matches the ground truth.
This is the human-readable view Kadaster can call on any deed.

In [ ]:
def explain(deed_id: str) -> pd.DataFrame:
    """Show orchestration decision for a single deed."""
    if deed_id not in test_ids:
        raise ValueError(f"{deed_id!r} not in test set")
    i = test_ids.index(deed_id)

    rows = []
    for j, code in enumerate(classes):
        b_score  = float(METHOD1_scores[i, j])
        r_score  = float(METHOD2_scores[i, j])
        method   = routing_map.get(code, METHOD1)
        routed_s = b_score if method == METHOD1 else r_score
        if routed_s < 0.02 and not y_true[i, j] and not y_pred_routed[i, j]:
            continue
        rows.append({
            "code":         code,
            "routed_to":    method,
            "METHOD1_score":   round(b_score,  4),
            "METHOD2_score":  round(r_score,  4),
            "routed_score": round(routed_s, 4),
            "predicted":    bool(y_pred_routed[i, j]),
            "true_label":   bool(y_true[i, j]),
        })

    result    = pd.DataFrame(rows).sort_values("routed_score", ascending=False).reset_index(drop=True)
    conf_row  = deed_conf_df[deed_conf_df["akteId"] == deed_id].iloc[0]
    escalated = bool(uncertain_mask[i])

    print(f"Deed              : {deed_id}")
    if bool(conf_row["no_pred"]):
        print(f"Weakest accepted  : none predicted")
    else:
        print(f"Weakest accepted  : {conf_row['min_pred_score']:.4f}   "
              f"Gap to strongest rejected: {conf_row['gap']:.4f}")
    print(f"Escalated         : {escalated}")
    print(f"Ground truth      : {[classes[j] for j in range(len(classes)) if y_true[i, j]]}")
    print()
    return result


# Demo on the first 3 test deeds
for did in test_ids[:3]:
    try:
        display(explain(did))
    except NameError:
        print(explain(did).to_string())
    print("-" * 60)

## 8. Write Orchestration Contract

Build `orchestration.parquet` — the shared contract file.
Escalated deeds have `predicted=False` for all codes (abstain; human handles them).
Scores still carry the routed confidence so downstream tools can inspect the reason.

In [ ]:
orch_pred = y_pred_routed.astype(bool).copy()
orch_pred[uncertain_mask, :] = False   # abstain on uncertain deeds

m_orch = evaluate(y_true, orch_pred.astype(np.int32), classes, method="orchestration")

# Auto-handled subset: deeds that were NOT escalated
auto_mask = ~uncertain_mask
m_auto          = evaluate(y_true[auto_mask], orch_pred[auto_mask].astype(np.int32),    classes, method="orchestration_auto")
m_auto_eurobert = evaluate(y_true[auto_mask], METHOD1_pred[auto_mask].astype(np.int32), classes, method=f"{METHOD1}_auto")

print()
print("Ablation: METHOD2 | METHOD1 | routed (no escalation) | orchestration (with escalation)")
print_summary([m_METHOD2, m_METHOD1, m_routed, m_orch])

write_predictions(test_ids, routed_scores, orch_pred, classes, method="orchestration")

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# FINAL RESULTS SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
W = 60
sep  = "=" * W
thin = "-" * W

print(sep)
print("  FINAL ORCHESTRATION RESULTS")
print(sep)

# ── 1. Routing ────────────────────────────────────────────────────────────────
print()
print("  ROUTING  (fit on val, applied to test)")
print(thin)
METHOD1_code_list  = sorted([c for c, m in routing_map.items() if m == METHOD1])
METHOD2_code_list = sorted([c for c, m in routing_map.items() if m != METHOD1])
print(f"  METHOD1  : {METHOD1_routes:2d} codes  → {METHOD1_code_list[:10]}{'...' if len(METHOD1_code_list) > 10 else ''}")
print(f"  METHOD2     : {METHOD2_routes:2d} codes  → {METHOD2_code_list}")

# ── 2. Disagreement ───────────────────────────────────────────────────────────
print()
print("  DISAGREEMENT SIGNAL  (METHOD1 vs METHOD2 on test)")
print(thin)
n_agree    = int(agree_mask.sum())
n_disagree = int(disagree_mask.sum())
fc_agree   = fully_correct[agree_mask].mean()
fc_disagree= fully_correct[disagree_mask].mean()
print(f"  Disagreement rate : {disagree_mask.mean():.1%}  ({n_disagree} / {len(test_ids)} deeds)")
print(f"  Agree    ({n_agree:4d} deeds) → fully-correct : {fc_agree:.1%}   error : {1-fc_agree:.1%}")
print(f"  Disagree ({n_disagree:4d} deeds) → fully-correct : {fc_disagree:.1%}   error : {1-fc_disagree:.1%}")

# ── 3. Method metrics ─────────────────────────────────────────────────────────
print()
print("  METHOD METRICS  (test set, no escalation)")
print(thin)
print(f"  {'Method':<22} {'micro-F1':>9}  {'macro-F1':>9}  {'active codes':>12}")
print(f"  {'------':<22} {'---------':>9}  {'---------':>9}  {'------------':>12}")
for m in [m_METHOD2, m_METHOD1, m_routed]:
    name   = m.get("method", "?")
    micro  = m.get("micro_f1",  m.get("micro-f1",  0.0))
    macro  = m.get("macro_f1",  m.get("macro-f1",  0.0))
    active = m.get("active_labels", m.get("active labels", "?"))
    print(f"  {name:<22} {micro:>9.4f}  {macro:>9.4f}  {str(active):>12}")

# ── 4. Escalation breakdown ───────────────────────────────────────────────────
print()
print("  ESCALATION BREAKDOWN  (test set)")
print(thin)
print(f"  AUTO_THRESHOLD (val-fitted) : {AUTO_THRESHOLD:.3f}  (weakest accepted code score)")
print(f"  MIN_GAP  (hardcoded)        : {MIN_GAP}  (accept/reject score separation)")
print()
print(f"  Escalated by no_pred only   : {cond_no_pred.sum():4d}  ({cond_no_pred.mean():.1%})")
print(f"  Escalated by low score only : {cond_low_score.sum():4d}  ({cond_low_score.mean():.1%})")
print(f"  Escalated by low gap only   : {cond_low_gap.sum():4d}  ({cond_low_gap.mean():.1%})")
print(f"  Escalated by score + gap    : {cond_multi.sum():4d}  ({cond_multi.mean():.1%})")
print(f"  ─────────────────────────────────────")
print(f"  Total escalated             : {uncertain_mask.sum():4d}  ({uncertain_mask.mean():.1%})")
print(f"  Auto-handled                : {(~uncertain_mask).sum():4d}  ({(~uncertain_mask).mean():.1%})")

# ── 5. Final performance ──────────────────────────────────────────────────────
print()
print("  ORCHESTRATION PERFORMANCE  (test set)")
print(thin)
orch_micro = m_orch.get("micro_f1", m_orch.get("micro-f1", 0.0))
orch_macro = m_orch.get("macro_f1", m_orch.get("macro-f1", 0.0))
auto_micro = m_auto.get("micro_f1", m_auto.get("micro-f1", 0.0))
auto_macro = m_auto.get("macro_f1", m_auto.get("macro-f1", 0.0))
eb_auto_micro = m_auto_eurobert.get("micro_f1", m_auto_eurobert.get("micro-f1", 0.0))
eb_auto_macro = m_auto_eurobert.get("macro_f1", m_auto_eurobert.get("macro-f1", 0.0))
print(f"  Overall  (abstained deeds count as wrong)")
print(f"    micro-F1 : {orch_micro:.4f}   macro-F1 : {orch_macro:.4f}")
print()
print(f"  Auto-handled subset only  ({(~uncertain_mask).sum()} deeds, {(~uncertain_mask).mean():.1%} coverage)")
print(f"  {'Method':<30} {'micro-F1':>9}  {'macro-F1':>9}")
print(f"  {'-'*30} {'-'*9}  {'-'*9}")
print(f"  {'EuroBERT (no escalation)':<30} {eb_auto_micro:>9.4f}  {eb_auto_macro:>9.4f}")
print(f"  {'Orchestration':<30} {auto_micro:>9.4f}  {auto_macro:>9.4f}")
print(f"  {'Delta (Orch - EuroBERT)':<30} {auto_micro-eb_auto_micro:>+9.4f}  {auto_macro-eb_auto_macro:>+9.4f}")
print()
print(sep)
